# **Implementación del Algoritmo de Shor para resolver el  DLP**

## Oráculo basado en matrices de permutación

### Rodrigo Hernández Sacristán

#### Máster en Computación Cuántica, Universidad Internacional de la Rioja (UNIR)

In [1]:
## LIBRERÍAS NECESARIAS 

import numpy as np
import math
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.library import UnitaryGate 
from qiskit.circuit.library import QFTGate
from qiskit_aer import AerSimulator
from qiskit import transpile
from qiskit.visualization import plot_histogram
from IPython.display import display
from qiskit.quantum_info import Operator
from qiskit.circuit.library import QFT
import warnings
import random

In [ ]:
def aplicar_oraculo_matrices(g, h, p, qc, reg_A, reg_B, reg_C, reg_anc=None):
    '''
    Implementa el oráculo cuántico para f(a,b) = g^a * h^(-b) mod p usando matrices de permutación
    '''
    
    num_qubits_reg = len(reg_A)
    q= 2**num_qubits_reg  # si  l=5 cúbits pues q=32
    
    # ----------------------------------------------------------------------
    # PASO 1: Inicializar el Registro C en |1> (ya que empezamos multiplicando)
    # ----------------------------------------------------------------------
    # El registro C por defecto está en |00000>, aplicamos una puerta X (NOT)
    # en el primer cúbit para obtener el estado clásico |1> (00001 en binario)
    qc.x(reg_C[0])
    
    # ----------------------------------------------------------------------
    # PASO 2: Calcular las potencias clásicas de las bases mod p
    # ----------------------------------------------------------------------
    # Registro A:
    factores_A = [pow(g, 2**i, p) for i in range(num_qubits_reg)]
    
    # Registro B: potencias del inverso de h. 
    h_inv = pow(h, -1, p)
    factores_B = [pow(h_inv, 2**j, p) for j in range(num_qubits_reg)]
    

    # -------------------------------------------------------------------------------------------------
    # PASO 3: Función interna para crear la matriz de multiplicación modular
    # -------------------------------------------------------------------------------------------------
    def crear_matriz_multiplicacion(k): 
        matriz = np.zeros((q, q)) # Matriz qxq rellena de 0s
        
        for x in range(q):
            if x < p: # Para estados válidos dentro de Z_p^*
                if x == 0:
                    estado_destino = 0
                else:
                    estado_destino = (x * k) % p
            else: 
                # Para  los estados que están fuera del grupo, los mapeamos a sí mismos
                # para mantener la matriz unitaria (identidad en la esquina inferior derecha)
                estado_destino = x
                
            matriz[estado_destino, x] = 1 # Matriz de permutación unitaria
            
        return UnitaryGate(matriz, label=f"x*{k} mod {p}")

    # ----------------------------------------------------------------------
    # PASO 4: Aplicar multiplicaciones controladas por el Registro A
    # ----------------------------------------------------------------------
    for i in range(num_qubits_reg):
        potencia = factores_A[i]
        
        # Optimización: si multiplicar por 1 no cambia el estado, nos saltamos la puerta
        if potencia == 1:
            continue
            
        puerta_mult = crear_matriz_multiplicacion(potencia)
        puerta_controlada = puerta_mult.control(1)
        
        # El cúbit de control es reg_A[i], y actúa sobre todo el reg_C
        qc.append(puerta_controlada, [reg_A[i]] + list(reg_C)) 
        
    # ----------------------------------------------------------------------
    # PASO 5: Aplicar multiplicaciones controladas por el Registro B
    # ----------------------------------------------------------------------
    for j in range(num_qubits_reg):
        potencia = factores_B[j]
        
        if potencia == 1:
            continue
            
        puerta_mult = crear_matriz_multiplicacion(potencia)
        puerta_controlada = puerta_mult.control(1)
        
        # El cúbit de control es reg_B[j], y actúa sobre todo el reg_C
        qc.append(puerta_controlada, [reg_B[j]] + list(reg_C))